# Q2: Are Zurichs Ice Hockey Fans responsible for graffitis?
Zurichs ice hockey Team ZSC Lions moved from the Hallenstadion in Oerlikon to the Swiss Life Arena in Altstetten back in 2022 (https://www.swisslifearena.ch/arena/geschichte/). This question focuses on determining wheter the number of reported graffitis has decreased in Oerlikon and increased in Altstetten after the local ice hockey team changed its home stadium in 2022. The idea is to derive the number of reported graffitis for both locations at two different time periods, one before and one after the the moving.

**Summarized workflow:**  

0. Import all packages needed
1. Import the files needed
2. Filter for the two neighborhoods needed
3. Filter for the category graffiti
4. Filter for the two timespanns needed 
5. Export the results as a table 
6. Plot the results and export them

## 0. Importing all packages

In [1]:
import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import cmcrameri.cm as cmc
from pathlib import Path
#from matplotlib_scalebar.scalebar import ScaleBar

import sys
sys.path.append("../scripts")

from functions_zh_wie_neu import import_preprocessed_data #type: ignore
from functions_zh_wie_neu import export_result_to_png #type: ignore

## 1. Import the data needed

In [4]:
quartiere=import_preprocessed_data("quartiere_processed.gpkg")
reports_complete=import_preprocessed_data("reports_quartiere_sjoin.gpkg")

## 2. Filter for the two neighborhoods needed
First, we need to determine which two neighborhoods are actually needed. Apparently, the names of the two statistical neighborhoods are Altstetten and Oerlikon (visually derived from: https://www.stadt-zuerich.ch/geodaten/download/Statistische_Quartiere?format=10005). 

In [15]:
# Filter the reports for these two neighborhoods:
reports_hockey=reports_complete[(reports_complete["qname"]=="Altstetten") | (reports_complete["qname"]=="Oerlikon")]

#check for the unique quartier names:
print(reports_hockey["qname"].unique()) #only Altstetten and Oerlikon present

<ArrowStringArray>
['Altstetten', 'Oerlikon']
Length: 2, dtype: str


In [16]:
#make a function out of this!!!
n_reports=(
    reports_complete
    .groupby("qname")
    .size()
    .reset_index(name="nr_reports")#this transforms the qnr to a data column named nr_reports (after the code above it was set as the index)
    .sort_values(by="nr_reports",ascending=False) #displays me the highest values first
)

n_reports_hockey=n_reports[(n_reports["qname"]=="Altstetten") | (n_reports["qname"]=="Oerlikon")]
n_reports_hockey.head(2)

#second check: how many rows are present:
if  reports_hockey.shape[0]==n_reports_hockey["nr_reports"].sum():
    print("Correct.")

Correct.


* Put the reports per neighborhood into a function and import it
* sturcture it nicely

## 3. Filter for the category graffiti
* add notes and describe code!!!

In [19]:
reports_hockey.head(2)

,service_request_id,requested_datetime,e,n,service_code,description,index_right,qnr,qname,knr,kname,geometry
7,9,2013-03-18 07:06:53,2678769,1248921,Strasse/Trottoir/Platz,"Feldblumens: Feldblumenstrasse 30, Schlagloch",28,92,Altstetten,9,Kreis 9,POINT (2678769 1248921)
8,10,2013-03-18 07:16:54,2678479,1248732,Strasse/Trottoir/Platz,Rautistrass: Rautistrasse / Friedhofstrasse: S...,28,92,Altstetten,9,Kreis 9,POINT (2678479 1248732)


In [23]:
reports_hockey=reports_hockey[reports_hockey["service_code"]=="Graffiti"]
reports_hockey.shape


(293, 12)

## 4. filter for time
add notes and do it